# Stage 11 V2d — Canonical CPU Notehead Semantic Rerun

This replaces the failed cross-host pixel-identity bridge. The frozen Restore model is executed on CPU and the **actual CPU outputs** are persisted and hash-bound. Notehead measurement uses only the pinned Oemer `seg_net` checkpoint through a cost-bounded ONNX CPU path; the full Oemer pipeline and `unet_big` are not executed. No GPU is required.

Cost control: after the first source+restored detector pair, the runner projects the 18-page detector time. If the projection exceeds 90 minutes it stops before the full rerun. Per-page artifacts are saved to Drive, and a 110-minute total wall-clock guard prevents an hours-long runaway.


In [ ]:
# 1) Build the exact isolated canonical CPU runtime.
import platform, subprocess, sys
from pathlib import Path
print('colab system python', platform.python_version())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.12.10'], check=True)
VENV = Path('/content/st-score-restore-canonical-semantic-py3135')
subprocess.run(['uv','venv','--python','3.13.5','--clear',str(VENV)], check=True)
PY = VENV/'bin'/'python'
subprocess.run(['uv','pip','install','--python',str(PY),'numpy>=2.0,<2.3','opencv-python-headless','onnxruntime==1.20.1','filelock','typing-extensions','sympy','networkx','jinja2','fsspec'], check=True)
subprocess.run(['uv','pip','install','--python',str(PY),'--index-url','https://download.pytorch.org/whl/cpu','torch==2.10.0'], check=True)
verify = r'''import platform, torch, onnxruntime as ort, cv2, numpy as np
print('python', platform.python_version())
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('ort', ort.__version__, ort.get_available_providers())
print('numpy', np.__version__, 'opencv', cv2.__version__)
assert platform.python_version() == '3.13.5'
assert torch.__version__ == '2.10.0+cpu'
assert not torch.cuda.is_available()
assert ort.__version__ == '1.20.1'
assert 'CPUExecutionProvider' in ort.get_available_providers()
assert tuple(map(int, np.__version__.split('.')[:2])) < (2, 3)
'''
subprocess.run([str(PY), '-c', verify], check=True)
print('EXACT CANONICAL CPU RUNTIME READY')


In [ ]:
# 2) Mount Drive, verify prior durable cache, and pin repository code to the reviewed runner commit.
from google.colab import drive
drive.mount('/content/drive')
import shutil, subprocess
from pathlib import Path
CACHE = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_CACHE_V1')
RESULTS = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS')
required = [CACHE/'cache_manifest.json', CACHE/'exact_inputs'/'v2a_candidate_512.torchscript.pt', CACHE/'oemer_checkpoints'/'seg_net'/'model.onnx', RESULTS/'v2d_colab_gpu_detector_benchmark_result.json']
for path in required:
    if not path.exists(): raise FileNotFoundError(path)
REPO = Path('/content/st-score-restore-engine')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git',str(REPO)], check=True)
EXPECTED_COMMIT = 'd88c11dbcc2688adeb6e2bb8f2e8d737cc4336cd'
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXPECTED_COMMIT], check=True)
actual = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
assert actual == EXPECTED_COMMIT
print('DRIVE CACHE + PINNED REPO READY', actual)


In [ ]:
# 3) Run CPU-only canonical semantic rerun. Durable per-page cache survives disconnects.
import os, subprocess
LOG = RESULTS/'v2d_canonical_cpu_notehead_semantic_rerun.log'
env = os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['PYTHONFAULTHANDLER']='1'; env['PYTHONPATH']=str(REPO/'src')
cmd = [str(PY), '-m', 'st_score_restore.stage11_v2d_canonical_cpu_notehead_semantic_rerun', '--run']
print('CPU ONLY — no GPU required')
print('running:', ' '.join(cmd))
print('log:', LOG)
with LOG.open('a', encoding='utf-8', buffering=1) as log:
    log.write('\n===== CANONICAL CPU NOTEHEAD SEMANTIC RERUN =====\n')
    proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end=''); log.write(line)
    rc = proc.wait()
if rc != 0: raise subprocess.CalledProcessError(rc, cmd)


## Expected markers

`EXACT CANONICAL CPU RUNTIME READY` → `DRIVE CACHE + PINNED REPO READY` → `CANONICAL CPU PREFLIGHT PASS` → `NOTEHEAD-ONLY DETECTOR PREFLIGHT PASS` → `CANARY COST BUDGET PASS` → `CANONICAL CPU NOTEHEAD SEMANTIC RERUN COMPLETE` → `SAVED:`

If `COST BUDGET` / `projected detector runtime exceeds cost budget` appears, stop: the runner intentionally refused an hours-long full run and preserved completed cache. Completion still keeps overall semantic preservation, production readiness, and Stage 12 authorization false until the result is reviewed.
